# Store Data - Simple Baseline

- [Forecasting with ML and Stuff](https://www.kaggle.com/code/ryanholbrook/forecasting-with-machine-learning/tutorial)
- [Kaggle Page](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)


## Imports

In [ ]:
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader

import mlflow
from common.git import get_branch, get_sha
from common.model_registry import TRACKING_URI
from time_series.store_sales import StoreData

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

## mlflow Setup

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)

mlflow.set_experiment("StoreSales_TransformerBasic")
mlflow.set_experiment_tags(
    {
        "dataset": "store-sales-kaggle",
        "task": "time-series-forecasting",
        "loss": "RMSLE",
        "prediction_horizon_days": "16",
    }
)

# Notes

From the information
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Load and Explore Data

In [ ]:
store_data = StoreData()
store_data.train

# Model Design

- Input data is essentially `date x store_nbr x family`, which is easily tensor-able. The overall data will be `batch, date_len, store_nbr, family`. 
  - Remember to use `batch_first=True` for any LSTM or whatever models. Apparently having the batch dimension be second is an old convention.
- We need to predict 2017-08-16 to 2017-08-31, so 16 day prediction window
- Input window length will be a parameter, sent to the dataset and model
- loss is Root-Mean-Squared-Logarithmic-Error (given by problem)
  - $\text{RMSE}(\log(1+\hat{y}), \log(1+y))$

In [ ]:
class StoreSalesTransformer(nn.Module):
    def __init__(self, n_output_steps: int = store_data.output_lags) -> None:
        super().__init__()
        self.n_output_steps = n_output_steps
        self.transformer = nn.Transformer(
            d_model=store_data.families.size * store_data.stores.shape[0],
            nhead=store_data.stores.shape[0] // 6,
            batch_first=True,
        )

    def forward(self, input_data: Tensor) -> Tensor:
        ...


model = StoreSalesTransformer()
model.transformer

In [ ]:
store_data_loader = DataLoader(store_data, batch_size=10)
store_data_loader

In [ ]:
batch_X, batch_y = next(iter(store_data_loader))

In [ ]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=store_data.families.size * store_data.stores.shape[0],
    nhead=9,
    batch_first=True,
    dtype=torch.float64,
)

encoder_input_X = batch_X.reshape(batch_X.shape[0], batch_X.shape[1], batch_X.shape[2] * batch_X.shape[3])
layer_output = encoder_layer(encoder_input_X)
layer_output.shape

# Training

In [ ]:
with mlflow.start_run(
    run_name="transformer-v1",
    tags={
        "architecture": "transformer",
        "input_window_days": str(store_data.window_lags),
        "output_window_days": str(store_data.output_lags),
        "device": str(device),
        "git_branch": get_branch(),
        "git_sha": get_sha(),
    }
) as mlflow_run:
    ...